# Logic

**Propositional logic** represents knowledge as sentences that are true or false, built from simple propositions with the connectives *not* (¬), *and* (∧), *or* (∨), *implies* (→) and *if and only if* (↔). A **knowledge base** is a set of sentences taken as true. It **entails** a sentence α, written KB ⊨ α, when α is true in **every** world in which the knowledge base is true.

## Sentences as Python objects

Each kind of sentence is a small class that can evaluate itself in a **model**, a dictionary that gives a truth value to every proposition, and can list the propositions it uses. The operators `~`, `&` and `|` build the sentences.

In [1]:
class Sentence:
    def __invert__(self):
        return Not(self)

    def __and__(self, other):
        return And(self, other)

    def __or__(self, other):
        return Or(self, other)


class Symbol(Sentence):
    def __init__(self, name):
        self.name = name

    def evaluate(self, model):
        return model[self.name]

    def symbols(self):
        return {self.name}

    def __repr__(self):
        return self.name


class Not(Sentence):
    def __init__(self, operand):
        self.operand = operand

    def evaluate(self, model):
        return not self.operand.evaluate(model)

    def symbols(self):
        return self.operand.symbols()

    def __repr__(self):
        return f"¬{self.operand}"


class Connective(Sentence):
    symbol = "?"

    def __init__(self, left, right):
        self.left, self.right = left, right

    def symbols(self):
        return self.left.symbols() | self.right.symbols()

    def __repr__(self):
        return f"({self.left} {self.symbol} {self.right})"


class And(Connective):
    symbol = "∧"

    def evaluate(self, model):
        return self.left.evaluate(model) and self.right.evaluate(model)


class Or(Connective):
    symbol = "∨"

    def evaluate(self, model):
        return self.left.evaluate(model) or self.right.evaluate(model)


class Implies(Connective):
    symbol = "→"

    def evaluate(self, model):
        return not self.left.evaluate(model) or self.right.evaluate(model)


class Iff(Connective):
    symbol = "↔"

    def evaluate(self, model):
        return self.left.evaluate(model) == self.right.evaluate(model)

## Entailment by model checking

The most direct way to decide entailment is **model checking**: list every possible world, and check that the query is true in each one where the knowledge base is. With *n* propositions there are 2ⁿ worlds, so it only works for small problems; real reasoners use inference rules, such as **resolution**, or SAT solvers.

A question then has three possible answers, not two: *yes* if the knowledge base entails it, *no* if it entails its negation, and *unknown* if it entails neither.

In [2]:
from itertools import product


def entails(knowledge_base, query):
    names = sorted(set().union(*(sentence.symbols() for sentence in knowledge_base + [query])))
    for values in product([False, True], repeat=len(names)):
        model = dict(zip(names, values))
        if all(sentence.evaluate(model) for sentence in knowledge_base) and not query.evaluate(model):
            return False
    return True


def answer(knowledge_base, query):
    if entails(knowledge_base, query):
        return "yes"
    if entails(knowledge_base, ~query):
        return "no"
    return "unknown"

## A whodunit

Exactly one of Alice, Bob and Carol is guilty. Alice would only have done it with Carol's help, and Carol has an alibi.

In [3]:
alice, bob, carol = Symbol("alice"), Symbol("bob"), Symbol("carol")

exactly_one_guilty = [alice | bob | carol, ~(alice & bob), ~(alice & carol), ~(bob & carol)]
clues = [Implies(alice, carol), ~carol]

for suspect in [alice, bob, carol]:
    print(f"Is {suspect} guilty? {answer(exactly_one_guilty + clues, suspect)}")

Is alice guilty? no
Is bob guilty? yes
Is carol guilty? no


Without Carol's alibi, the knowledge base still rules out Alice, but it can't decide between Bob and Carol:

In [4]:
for suspect in [alice, bob, carol]:
    print(f"Is {suspect} guilty? {answer(exactly_one_guilty + [Implies(alice, carol)], suspect)}")

Is alice guilty? no
Is bob guilty? unknown
Is carol guilty? unknown


*Unknown* is an honest answer that a yes-or-no function would hide: a program that answered *no* here would be asserting Bob's innocence without grounds.

## Logic programming

Propositional logic deals with fixed statements: *alice is guilty*. **First-order logic** adds objects, relations between them and variables: *for every X, Y and Z, if X is a parent of Y and Y is an ancestor of Z, then X is an ancestor of Z*. One sentence then covers every family.

**Logic programming** runs such sentences as programs: you state facts and rules, ask a question, and the language searches for the answers. Its main language is **Prolog** (Alain Colmerauer and Philippe Roussel, 1972). Lisp (John McCarthy, 1958), the other classic language of AI, is not a logic language: it is functional, and symbolic AI used it to write programs such as expert systems.

A family in Prolog. A name in lower case is a constant, a capitalised one a variable, `:-` reads *if* and the comma *and*:

```prolog
parent(ana, bea).
parent(bea, carla).
parent(carla, dani).
parent(ana, eva).

ancestor(X, Y) :- parent(X, Y).
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).
```

Three questions, with the answers of SWI-Prolog, the most widely used free implementation:

```prolog
?- findall(Who, ancestor(ana, Who), Descendants).
Descendants = [bea, eva, carla, dani].

?- findall(Who, ancestor(Who, dani), Ancestors).
Ancestors = [carla, ana, bea].

?- ancestor(eva, dani).
false.
```

You can run them, without installing anything, in [SWISH](https://swish.swi-prolog.org), SWI-Prolog's online environment.

The same two rules answer opposite questions, who descends from Ana and whom Dani descends from, because a Prolog program states relations, not procedures. Prolog finds the answers by **backward chaining**, MYCIN's strategy in [the expert system](rules_and_expert_systems.ipynb): it tries the clauses in the order they are written, depth first, which is why the answers come in that order.

`false` means *not provable from these facts*. Prolog assumes that whatever it can't prove is false, the **closed-world assumption**, where the model checker above would have answered *unknown*: an absent fact counts as a false one, as an absent row does in a database.

Prolog is a niche language today, used where rules are the natural description of a problem: IBM's Watson, for example, analysed the questions of the quiz show *Jeopardy!* with Prolog rules when it won in 2011. Its simpler relative **Datalog** found a second life in tools that analyse source code: GitHub's CodeQL queries a codebase in a dialect of Datalog.

## Recursive rules in SQL

The idea that reached every developer is recursion over relations. Since the SQL:1999 standard, SQL databases answer **recursive queries**, which compute the same *ancestor* relation from a table of parents. SQLite, which comes with Python, runs one:

In [5]:
import sqlite3

connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE parent (older TEXT, younger TEXT)")
connection.executemany("INSERT INTO parent VALUES (?, ?)",
                       [("ana", "bea"), ("bea", "carla"), ("carla", "dani"), ("ana", "eva")])

In [6]:
ancestors_query = """
WITH RECURSIVE ancestor(older, younger) AS (
    SELECT older, younger FROM parent
    UNION
    SELECT parent.older, ancestor.younger
    FROM parent JOIN ancestor ON parent.younger = ancestor.older
)
SELECT older, younger FROM ancestor ORDER BY older, younger
"""

for older, younger in connection.execute(ancestors_query):
    print(f"{older} is an ancestor of {younger}")

ana is an ancestor of bea
ana is an ancestor of carla
ana is an ancestor of dani
ana is an ancestor of eva
bea is an ancestor of carla
bea is an ancestor of dani
carla is an ancestor of dani


The two parts of the query are the two Prolog rules: the first `SELECT` says that a parent is an ancestor, the second that a parent of an ancestor is an ancestor. The database evaluates it by **forward chaining**: each pass joins the parents with the pairs found so far, until a pass adds nothing new, the fixed point of [the expert system](rules_and_expert_systems.ipynb). Recursive queries are how databases walk organisation charts, folder trees and the parts lists of manufactured products.